## Imports

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    trim,
    to_date,
    current_timestamp,
    row_number,
    lit,
    when
)
from pyspark.sql.window import Window

## Bronze table

In [0]:
bronze_df = spark.table("bronze_customers")

display(bronze_df.limit(10))
print(f"Bronze rows: {bronze_df.count():,}")

customer_id,first_name,last_name,email,phone,date_of_birth,gender,signup_date,status,created_at,updated_at,source_system
CRM0000001,Angel,Hill,angel.hill1@example.com,0318196001,1959-03-26,male,2024-08-26,active,2024-08-26,2025-04-11,crm
CRM0000002,Anthony,Gonzalez,anthony.gonzalez2@example.com,+61-486-379-402,1989-06-09,male,2023-12-25,inactive,2023-12-25,2026-05-07,crm
CRM0000003,Eric,Bernard,eric.bernard3@example.com,(02).1615.5940,1991-06-06,male,2023-02-26,inactive,2023-02-26,2026-03-18,crm
CRM0000004,Teresa,Gray,teresa.gray4@example.com,0449-593-103,1994-02-28,other,2024-02-19,active,2024-02-19,2026-07-03,crm
CRM0000005,Melissa,Delacruz,melissa.delacruz5@example.com,+61 7 7525 5341,1983-10-31,other,2026-05-19,active,2026-05-19,2026-11-24,crm
CRM0000006,Jessica,Martin,jessica.martin6@example.com,+61 7 8350 3056,1962-04-28,other,2022-04-23,blocked,2022-04-23,2024-11-18,crm
CRM0000007,Sherri,Walter,sherri.walter7@example.com,(03).7672.4238,1979-05-31,other,2022-07-05,active,2022-07-05,2025-09-29,crm
CRM0000008,Ross,Silva,ross.silva8@example.com,(03)28710122,1995-05-11,other,2025-03-10,inactive,2025-03-10,2026-06-29,crm
CRM0000009,Bryan,James,bryan.james9@example.com,+61.7.8018.4514,1972-11-05,male,2023-07-26,inactive,2023-07-26,2025-08-22,crm
CRM0000010,Stephanie,Liu,stephanie.liu10@example.com,(03) 8148 9325,1993-06-30,female,2023-09-21,blocked,2023-09-21,2026-09-18,crm


Bronze rows: 102,000


## Silver transformation

In [0]:
VALID_STATUSES = ["active", "inactive", "blocked", "pending"]

standardised_df = (
    bronze_df
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("gender", lower(trim(col("gender"))))
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("date_of_birth", to_date(col("date_of_birth")))
    .withColumn("signup_date", to_date(col("signup_date")))
    .withColumn("created_at", to_date(col("created_at")))
    .withColumn("updated_at", to_date(col("updated_at")))
    .withColumn("_silver_processed_at", current_timestamp())
)

window_spec = Window.partitionBy("customer_id").orderBy(
    col("updated_at").desc_nulls_last()
)

deduped_df = (
    standardised_df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

quality_df = (
    deduped_df
    .withColumn(
        "_has_future_updated_at",
        col("updated_at") > current_timestamp()
    )
    .withColumn(
        "_is_valid",
        when(col("customer_id").isNull(), lit(False))
        .when(~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), lit(False))
        .when(~col("status").isin(VALID_STATUSES), lit(False))
        .otherwise(lit(True))
    )
    .withColumn(
        "_rejection_reason",
        when(col("customer_id").isNull(), lit("customer_id is null"))
        .when(~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"), lit("invalid email"))
        .when(~col("status").isin(VALID_STATUSES), lit("invalid status"))
        .otherwise(lit(None))
    )
)

valid_customers = quality_df.filter(col("_is_valid") == True)
rejected_customers = quality_df.filter(col("_is_valid") == False)

## Delta tables

In [0]:
(
    valid_customers
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver_customers")
)

(
    rejected_customers
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("rejected_customers")
)

print("Silver Customers completed")
print(f"Bronze rows: {bronze_df.count():,}")
print(f"After deduplication: {deduped_df.count():,}")
print(f"Valid customers: {valid_customers.count():,}")
print(f"Rejected customers: {rejected_customers.count():,}")

Silver Customers completed
Bronze rows: 102,000
After deduplication: 100,000
Valid customers: 99,048
Rejected customers: 952


In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|  bronze_customers|      false|
| default|rejected_customers|      false|
| default|  silver_customers|      false|
+--------+------------------+-----------+

